In [9]:
import numpy as np
import wave
import hashlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from google.colab import files


# ============================================================
# CONFIGURATION
# ============================================================

SIGMA = 10.0
RHO = 28.0
BETA = 8.0 / 3.0

X0 = 1.0
Y0 = 1.0
Z0 = 1.0

DT = 0.01

# Number of initial Lorenz iterations to discard.
# This removes the initial transient before generating keys.
WARMUP_STEPS = 1000


# ============================================================
# LORENZ ATTRACTOR
# ============================================================

def lorenz_step(x, y, z):

    dx = SIGMA * (y - x)
    dy = x * (RHO - z) - y
    dz = x * y - BETA * z

    x_new = x + dx * DT
    y_new = y + dy * DT
    z_new = z + dz * DT

    return x_new, y_new, z_new


# ============================================================
# GENERATE LORENZ SEQUENCE
# ============================================================

def generate_lorenz_sequence(num_samples):

    x = X0
    y = Y0
    z = Z0

    # --------------------------------------------------------
    # Warm-up phase
    # --------------------------------------------------------

    for _ in range(WARMUP_STEPS):

        x, y, z = lorenz_step(
            x, y, z
        )

    # --------------------------------------------------------
    # Allocate arrays
    # --------------------------------------------------------

    x_values = np.zeros(
        num_samples,
        dtype=np.float64
    )

    y_values = np.zeros(
        num_samples,
        dtype=np.float64
    )

    z_values = np.zeros(
        num_samples,
        dtype=np.float64
    )

    # --------------------------------------------------------
    # Generate chaotic sequence
    # --------------------------------------------------------

    for i in range(num_samples):

        x, y, z = lorenz_step(
            x, y, z
        )

        x_values[i] = x
        y_values[i] = y
        z_values[i] = z

    return x_values, y_values, z_values


# ============================================================
# CONVERT CHAOTIC FLOATS TO 16-BIT VALUES
# ============================================================

def lorenz_to_16bit(values):

    # --------------------------------------------------------
    # Take absolute value
    # --------------------------------------------------------

    absolute_values = np.abs(values)

    # --------------------------------------------------------
    # Extract fractional information
    #
    # Multiplying by a large factor allows us to extract
    # significantly more changing information from the
    # floating-point chaotic sequence.
    # --------------------------------------------------------

    scaled = np.floor(
        absolute_values * 1_000_000_000_000
    )

    # --------------------------------------------------------
    # Convert to unsigned 64-bit first
    # --------------------------------------------------------

    scaled_uint64 = scaled.astype(
        np.uint64
    )

    # --------------------------------------------------------
    # Keep only the lower 16 bits
    # --------------------------------------------------------

    values_16 = (
        scaled_uint64 & 0xFFFF
    ).astype(np.uint16)

    return values_16


# ============================================================
# GENERATE COMBINED CHAOTIC KEY
# ============================================================

def generate_chaotic_key(
    x_values,
    y_values,
    z_values
):

    # Convert each Lorenz component to 16-bit
    x_key = lorenz_to_16bit(
        x_values
    )

    y_key = lorenz_to_16bit(
        y_values
    )

    z_key = lorenz_to_16bit(
        z_values
    )

    # --------------------------------------------------------
    # Combine X, Y and Z using XOR
    # --------------------------------------------------------

    xy_key = np.bitwise_xor(
        x_key,
        y_key
    )

    chaotic_key = np.bitwise_xor(
        xy_key,
        z_key
    )

    return (
        x_key,
        y_key,
        z_key,
        chaotic_key
    )


# ============================================================
# AUDIO VALIDATION
# ============================================================

def validate_audio(filename):

    # --------------------------------------------------------
    # Check extension
    # --------------------------------------------------------

    if not filename.lower().endswith(".wav"):

        raise ValueError(
            "Invalid file format.\n"
            "Please upload a WAV audio file."
        )

    # --------------------------------------------------------
    # Open WAV file
    # --------------------------------------------------------

    try:

        with wave.open(
            filename,
            "rb"
        ) as audio:

            channels = audio.getnchannels()
            sample_width = audio.getsampwidth()
            sample_rate = audio.getframerate()
            num_frames = audio.getnframes()
            compression = audio.getcomptype()
            compression_name = audio.getcompname()

    except wave.Error as error:

        raise ValueError(
            f"Unable to read WAV file: {error}"
        )

    # --------------------------------------------------------
    # Display information
    # --------------------------------------------------------

    print("\n========================================")
    print("          AUDIO INFORMATION")
    print("========================================")

    print(
        "File:",
        filename
    )

    print(
        "Channels:",
        channels
    )

    print(
        "Bit depth:",
        sample_width * 8,
        "bits"
    )

    print(
        "Sample rate:",
        sample_rate,
        "Hz"
    )

    print(
        "Number of frames:",
        num_frames
    )

    print(
        "Compression:",
        compression
    )

    print(
        "Compression name:",
        compression_name
    )

    # --------------------------------------------------------
    # Validate channels
    # --------------------------------------------------------

    if channels not in [1, 2]:

        raise ValueError(
            f"Unsupported number of channels: {channels}.\n"
            "Only mono and stereo WAV files are supported."
        )

    # --------------------------------------------------------
    # Validate PCM
    # --------------------------------------------------------

    if compression != "NONE":

        raise ValueError(
            "Unsupported audio encoding.\n"
            "Please upload an uncompressed PCM WAV file."
        )

    # --------------------------------------------------------
    # Validate 16-bit
    # --------------------------------------------------------

    if sample_width != 2:

        raise ValueError(
            f"Unsupported bit depth: "
            f"{sample_width * 8}-bit.\n\n"
            "This implementation requires "
            "16-bit PCM WAV audio."
        )

    print("\n✓ WAV format confirmed")
    print("✓ Uncompressed PCM confirmed")
    print("✓ 16-bit depth confirmed")

    if channels == 1:

        print("✓ Mono audio confirmed")

    else:

        print("✓ Stereo audio confirmed")

    print("✓ Audio validation successful")

    return (
        channels,
        sample_rate,
        num_frames
    )


# ============================================================
# READ AUDIO
# ============================================================

def read_audio(filename):

    with wave.open(
        filename,
        "rb"
    ) as audio:

        channels = audio.getnchannels()

        sample_rate = audio.getframerate()

        frames = audio.readframes(
            audio.getnframes()
        )

    # --------------------------------------------------------
    # Convert raw bytes to signed 16-bit samples
    # --------------------------------------------------------

    samples = np.frombuffer(
        frames,
        dtype=np.int16
    ).copy()

    return (
        samples,
        channels,
        sample_rate
    )


# ============================================================
# WRITE AUDIO
# ============================================================

def write_audio(
    filename,
    samples,
    channels,
    sample_rate
):

    with wave.open(
        filename,
        "wb"
    ) as audio:

        audio.setnchannels(
            channels
        )

        # 16-bit PCM = 2 bytes
        audio.setsampwidth(2)

        audio.setframerate(
            sample_rate
        )

        audio.writeframes(
            samples.astype(
                np.int16
            ).tobytes()
        )


# ============================================================
# ENCRYPT AUDIO
# ============================================================

def encrypt_audio(
    samples,
    chaotic_key
):

    # --------------------------------------------------------
    # Convert signed int16 to unsigned 16-bit representation
    # --------------------------------------------------------

    samples_uint = samples.view(
        np.uint16
    )

    # --------------------------------------------------------
    # XOR every audio sample with the chaotic key
    # --------------------------------------------------------

    encrypted_uint = np.bitwise_xor(
        samples_uint,
        chaotic_key
    )

    # --------------------------------------------------------
    # Convert back to signed int16
    # --------------------------------------------------------

    encrypted_samples = encrypted_uint.view(
        np.int16
    )

    return encrypted_samples


# ============================================================
# DECRYPT AUDIO
# ============================================================

def decrypt_audio(
    encrypted_samples,
    chaotic_key
):

    encrypted_uint = encrypted_samples.view(
        np.uint16
    )

    # XOR again with the SAME key
    decrypted_uint = np.bitwise_xor(
        encrypted_uint,
        chaotic_key
    )

    decrypted_samples = decrypted_uint.view(
        np.int16
    )

    return decrypted_samples


# ============================================================
# SHA-256
# ============================================================

def calculate_hash(samples):

    return hashlib.sha256(
        samples.tobytes()
    ).hexdigest()


# ============================================================
# CORRELATION
# ============================================================

def calculate_correlation(
    original,
    encrypted
):

    return np.corrcoef(
        original.astype(np.float64),
        encrypted.astype(np.float64)
    )[0, 1]


# ============================================================
# DISPLAY TEST VALUES
# ============================================================

def display_test_values(
    original_samples,
    x_key,
    y_key,
    z_key,
    chaotic_key,
    encrypted_samples,
    decrypted_samples
):

    print("\n========================================")
    print("       FIRST 10 TEST VALUES")
    print("========================================")

    number_of_values = min(
        10,
        len(original_samples)
    )

    for i in range(
        number_of_values
    ):

        print(
            f"Sample {i:02d} | "
            f"Original = {original_samples[i]:6d} | "
            f"X16 = {x_key[i]:5d} | "
            f"Y16 = {y_key[i]:5d} | "
            f"Z16 = {z_key[i]:5d} | "
            f"KEY = {chaotic_key[i]:5d} | "
            f"Encrypted = {encrypted_samples[i]:6d} | "
            f"Decrypted = {decrypted_samples[i]:6d}"
        )


# ============================================================
# WAVEFORM COMPARISON
# ============================================================

def plot_waveforms(
    original_samples,
    encrypted_samples,
    decrypted_samples
):

    # Limit display to first 5000 samples
    display_samples = min(
        5000,
        len(original_samples)
    )

    x = np.arange(
        display_samples
    )

    # --------------------------------------------------------
    # Original waveform
    # --------------------------------------------------------

    plt.figure(
        figsize=(14, 4)
    )

    plt.plot(
        x,
        original_samples[
            :display_samples
        ]
    )

    plt.title(
        "Original Audio Waveform"
    )

    plt.xlabel(
        "Sample"
    )

    plt.ylabel(
        "Amplitude"
    )

    plt.grid(
        True
    )

    plt.show()

    # --------------------------------------------------------
    # Encrypted waveform
    # --------------------------------------------------------

    plt.figure(
        figsize=(14, 4)
    )

    plt.plot(
        x,
        encrypted_samples[
            :display_samples
        ]
    )

    plt.title(
        "Encrypted Audio Waveform"
    )

    plt.xlabel(
        "Sample"
    )

    plt.ylabel(
        "Amplitude"
    )

    plt.grid(
        True
    )

    plt.show()

    # --------------------------------------------------------
    # Decrypted waveform
    # --------------------------------------------------------

    plt.figure(
        figsize=(14, 4)
    )

    plt.plot(
        x,
        decrypted_samples[
            :display_samples
        ]
    )

    plt.title(
        "Decrypted Audio Waveform"
    )

    plt.xlabel(
        "Sample"
    )

    plt.ylabel(
        "Amplitude"
    )

    plt.grid(
        True
    )

    plt.show()


# ============================================================
# UPLOAD AUDIO
# ============================================================

print("========================================")
print("     CHAOTIC AUDIO ENCRYPTION SYSTEM")
print("========================================")

print(
    "\nUpload a 16-bit PCM WAV audio file:"
)

uploaded = files.upload()


# ============================================================
# CHECK UPLOAD
# ============================================================

if len(uploaded) == 0:

    raise ValueError(
        "No file was uploaded."
    )

if len(uploaded) > 1:

    raise ValueError(
        "Please upload only ONE WAV file."
    )


filename = list(
    uploaded.keys()
)[0]

print(
    "\nUploaded file:",
    filename
)


# ============================================================
# VALIDATE AUDIO
# ============================================================

channels, sample_rate, num_frames = (
    validate_audio(filename)
)


# ============================================================
# READ AUDIO
# ============================================================

original_samples, channels, sample_rate = (
    read_audio(filename)
)


print("\n========================================")
print("             AUDIO LOADED")
print("========================================")

print(
    "Total 16-bit samples:",
    len(original_samples)
)

print(
    "Audio channels:",
    channels
)

print(
    "Sample rate:",
    sample_rate,
    "Hz"
)


# ============================================================
# GENERATE LORENZ SEQUENCE
# ============================================================

print(
    "\nGenerating Lorenz chaotic sequence..."
)

x_values, y_values, z_values = (
    generate_lorenz_sequence(
        len(original_samples)
    )
)

print(
    "✓ Lorenz sequence generated"
)

print(
    "✓ Warm-up steps:",
    WARMUP_STEPS
)


# ============================================================
# GENERATE CHAOTIC KEY
# ============================================================

print(
    "\nGenerating combined chaotic key..."
)

x_key, y_key, z_key, chaotic_key = (
    generate_chaotic_key(
        x_values,
        y_values,
        z_values
    )
)

print(
    "✓ X converted to 16-bit"
)

print(
    "✓ Y converted to 16-bit"
)

print(
    "✓ Z converted to 16-bit"
)

print(
    "✓ X XOR Y XOR Z completed"
)

print(
    "✓ Final 16-bit chaotic key generated"
)


# ============================================================
# DISPLAY LORENZ AND KEY VALUES
# ============================================================

print("\n========================================")
print("        LORENZ KEY GENERATION")
print("========================================")

for i in range(
    min(5, len(x_values))
):

    print(
        f"\nStep {i}"
    )

    print(
        f"X = {x_values[i]:.8f}"
    )

    print(
        f"Y = {y_values[i]:.8f}"
    )

    print(
        f"Z = {z_values[i]:.8f}"
    )

    print(
        f"X16 = {x_key[i]}"
    )

    print(
        f"Y16 = {y_key[i]}"
    )

    print(
        f"Z16 = {z_key[i]}"
    )

    print(
        f"Final Key = {chaotic_key[i]}"
    )


# ============================================================
# ENCRYPT
# ============================================================

print(
    "\nEncrypting audio..."
)

encrypted_samples = encrypt_audio(
    original_samples,
    chaotic_key
)

print(
    "✓ Every 16-bit audio sample encrypted"
)


# ============================================================
# SAVE ENCRYPTED AUDIO
# ============================================================

encrypted_filename = (
    "encrypted_audio.wav"
)

write_audio(
    encrypted_filename,
    encrypted_samples,
    channels,
    sample_rate
)

print(
    "✓ Encrypted file saved:",
    encrypted_filename
)


# ============================================================
# DECRYPT
# ============================================================

print(
    "\nDecrypting audio..."
)

decrypted_samples = decrypt_audio(
    encrypted_samples,
    chaotic_key
)

print(
    "✓ Decryption completed"
)


# ============================================================
# SAVE DECRYPTED AUDIO
# ============================================================

decrypted_filename = (
    "decrypted_audio.wav"
)

write_audio(
    decrypted_filename,
    decrypted_samples,
    channels,
    sample_rate
)

print(
    "✓ Decrypted file saved:",
    decrypted_filename
)


# ============================================================
# DISPLAY SAMPLE TEST VALUES
# ============================================================

display_test_values(
    original_samples,
    x_key,
    y_key,
    z_key,
    chaotic_key,
    encrypted_samples,
    decrypted_samples
)


# ============================================================
# HASH VERIFICATION
# ============================================================

original_hash = calculate_hash(
    original_samples
)

decrypted_hash = calculate_hash(
    decrypted_samples
)


# ============================================================
# SAMPLE VERIFICATION
# ============================================================

samples_match = np.array_equal(
    original_samples,
    decrypted_samples
)

hash_match = (
    original_hash == decrypted_hash
)


# ============================================================
# CORRELATION
# ============================================================

correlation = calculate_correlation(
    original_samples,
    encrypted_samples
)


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n========================================")
print("             FINAL RESULTS")
print("========================================")

print(
    "\nOriginal SHA-256:"
)

print(
    original_hash
)

print(
    "\nDecrypted SHA-256:"
)

print(
    decrypted_hash
)

print(
    "\nSample-by-sample match:",
    samples_match
)

print(
    "SHA-256 match:",
    hash_match
)

print(
    "Original/Encrypted correlation:",
    correlation
)


if samples_match and hash_match:

    print("\n========================================")
    print("              ✓ SUCCESS")
    print("========================================")

    print(
        "Decrypted audio is EXACTLY "
        "identical to the original."
    )

else:

    print("\n========================================")
    print("              ✗ ERROR")
    print("========================================")

    print(
        "Decrypted audio does NOT "
        "match the original."
    )


# ============================================================
# WAVEFORM COMPARISON
# ============================================================

print(
    "\nGenerating waveform comparison..."
)

plot_waveforms(
    original_samples,
    encrypted_samples,
    decrypted_samples
)


# ============================================================
# AUDIO PLAYBACK
# ============================================================

print("\n========================================")
print("             AUDIO PLAYBACK")
print("========================================")

print(
    "\nOriginal Audio:"
)

display(
    Audio(
        filename
    )
)

print(
    "\nEncrypted Audio:"
)

display(
    Audio(
        encrypted_filename
    )
)

print(
    "\nDecrypted Audio:"
)

display(
    Audio(
        decrypted_filename
    )
)


# ============================================================
# DOWNLOAD FILES
# ============================================================

print(
    "\nPreparing files for download..."
)

files.download(
    encrypted_filename
)

files.download(
    decrypted_filename
)

print(
    "\n✓ Complete processing finished."
)

Output hidden; open in https://colab.research.google.com to view.